# Man of the Match team predictor

**Author:** Kelvin Wilch

## The question

> For each finished 2018 or 2022 World Cup match, look at both teams' match statistics and pick **which of the two teams** the Man of the Match came from.

What this project does **not** do:

- It does not name the player. It only picks the team.
- It does not forecast before kick-off. It uses statistics that exist only once the match is over.
- It does not ask "did the winning team get the award?". Who won is never part of the label.

## How the data is shaped

`world_cup_team_statistics.csv` has **256 rows and 45 columns**. Every match appears as **two rows**, one per team. The column `Man of the Match` is `1` on the row of the team that supplied the award winner and `0` on the other row.

So the real sample size is **128 matches**, not 256 independent rows. The two rows of a match are two views of one decision. That single idea drives most of the choices below: I split the data by match, I cross-validate by match, and I score by match.

## How this notebook is organised

| Phase | What happens | Why it matters |
|---|---|---|
| 1. Data audit | Load the CSV and check its structure | A model trained on broken data gives broken answers |
| 2. Features and split | Choose the input columns, then set aside 26 test matches | The test matches must stay unseen until the end |
| 3. Baseline | Score a hand-written rule with no training | It is the bar a model has to beat |
| 4. Training and tuning | Fit three models and choose their settings by cross-validation | Settings are chosen without touching the test matches |
| 5. Evaluation | Score everything once on the 26 test matches | One fair, final comparison |

## How to read it

- Every section opens with a note that says **what** the section does and **why**.
- Every code cell has inline comments that explain the reason for each step.
- Cells titled **Self-check** run a function on tiny hand-made matches where I already know the right answer. If a check fails, the notebook stops, so a mistake cannot quietly reach the results.

## Setup: imports and constants

**What:** load the libraries and define the three constants the whole notebook shares.

**Why:** keeping every import in one cell shows at a glance what the project depends on. Naming the constants once (instead of retyping `"Man of the Match"` or `42` everywhere) means a typo cannot make two cells disagree.

In [1]:
# --- Data handling -----------------------------------------------------------
import numpy as np   # fast maths on arrays of numbers
import pandas as pd  # tables (DataFrames), like a spreadsheet in code

# --- scikit-learn: the machine learning toolkit ------------------------------
from sklearn.base import clone                        # makes a fresh, untrained copy of a model
from sklearn.impute import SimpleImputer              # fills in missing values
from sklearn.inspection import permutation_importance # measures how much a model relies on each stat
from sklearn.linear_model import LogisticRegression   # model 1: a weighted sum turned into a probability
from sklearn.model_selection import (
    GroupKFold,        # cross-validation that keeps a group (here: a match) together
    ParameterGrid,     # lists every combination of settings to try
    train_test_split,  # splits data into a training part and a test part
)
from sklearn.neighbors import KNeighborsClassifier    # model 3: the k most similar rows vote
from sklearn.pipeline import Pipeline                 # chains preparation steps and a model into one object
from sklearn.preprocessing import StandardScaler      # puts every stat on the same scale
from sklearn.tree import DecisionTreeClassifier       # model 2: a flowchart of yes/no questions

# --- Constants shared by every phase -----------------------------------------
# A relative path: the CSV sits next to this notebook, so it runs on any machine.
DATASET_PATH = "world_cup_team_statistics.csv"

# The label column: 1 = the award winner came from this team, 0 = from the opponent.
TARGET = "Man of the Match"

# A fixed seed makes every "random" step (the split, the coin flip, the tree)
# give the same result on every run, so the numbers in this notebook can be reproduced.
RANDOM_STATE = 42

# Show the full text of every table cell. Without this, pandas cuts long
# values short with "...", which would hide part of a model's settings.
pd.set_option("display.max_colwidth", None)

---
# Phase 1: Data audit

Before any modelling I check that the file really has the shape I think it has. Every later phase relies on "two rows per match, exactly one of them labelled 1".

## 1.1 Load the data and look at it

**What:** read the CSV into a table and print its size, the number of matches, the rows per tournament and the label counts.

**Why:** these four numbers are the facts the rest of the project is built on. If the file ever changes, this cell is where I would notice first.

In [2]:
# Read the CSV into a DataFrame: one row per team per match.
df = pd.read_csv(DATASET_PATH)

# .shape is (rows, columns). I expect (256, 45).
print("Rows, columns:", df.shape)

# .nunique() counts distinct values. Each match has one match_id, so this
# is the number of matches. I expect 128.
print("Matches:", df["match_id"].nunique())

# Rows per tournament. Each match gives 2 rows, so 128 rows = 64 matches.
print("\nTeam rows per tournament year:")
print(df["year"].value_counts().sort_index())

# How many rows carry each label. One 1 and one 0 per match means the two
# counts must be equal: the target is balanced by construction.
print("\nLabel counts (1 = this team supplied the Man of the Match):")
print(df[TARGET].value_counts().sort_index())

# The first rows. Rows 0 and 1 are the two sides of the same match.
display(df.head())

Rows, columns: (256, 45)
Matches: 128

Team rows per tournament year:
year
2018    128
2022    128
Name: count, dtype: int64

Label counts (1 = this team supplied the Man of the Match):
Man of the Match
0    128
1    128
Name: count, dtype: int64


,Date,Team,Opponent,Goal Scored,Ball Possession %,Attempts,On-Target,Off-Target,Blocked,Corners,...,award_winner_player_id,award_winner_team,fifa_report_url,possession_reported_pct,possession_in_contest_pct,incomplete_attempts,on_target_fifa_summary,on_target_shot_outcomes,statistics_method,event_source
0,2018-06-14,Russia,Saudi Arabia,5,40.0,13,7,3,3,6,...,NaN,Russia,NaN,40.0,NaN,NaN,NaN,NaN,Supplied 2018 statistics,Fjelstul worldcup v1.1.0 goals.csv and bookings.csv
1,2018-06-14,Saudi Arabia,Russia,0,60.0,6,0,3,3,2,...,NaN,Russia,NaN,60.0,NaN,NaN,NaN,NaN,Supplied 2018 statistics,Fjelstul worldcup v1.1.0 goals.csv and bookings.csv
2,2018-06-15,Egypt,Uruguay,0,43.0,8,3,3,2,0,...,NaN,Uruguay,NaN,43.0,NaN,NaN,NaN,NaN,Supplied 2018 statistics,Fjelstul worldcup v1.1.0 goals.csv and bookings.csv
3,2018-06-15,Uruguay,Egypt,1,57.0,14,4,6,4,5,...,NaN,Uruguay,NaN,57.0,NaN,NaN,NaN,NaN,Supplied 2018 statistics,Fjelstul worldcup v1.1.0 goals.csv and bookings.csv
4,2018-06-15,Iran,Morocco,1,36.0,8,2,5,1,2,...,NaN,Iran,NaN,36.0,NaN,NaN,NaN,NaN,Supplied 2018 statistics,Fjelstul worldcup v1.1.0 goals.csv and bookings.csv


## 1.2 Check the structure of every match

**What:** a list of `assert` statements. Each one states a rule the data must obey, and stops the notebook with an error if the rule is broken.

**Why:** the whole method assumes each match has exactly two mirrored rows and exactly one label of `1`. If even one match broke that rule (a missing row, two winners, a duplicated team), the match-level scoring later would be wrong without any visible error. An assert turns a silent problem into a loud one.

In [3]:
# --- The key columns must have no gaps ---------------------------------------
assert df["match_id"].notna().all()  # every row belongs to a known match
assert df["Team"].notna().all()      # every row names its team
assert df["Opponent"].notna().all()  # every row names the opponent
assert df[TARGET].notna().all()      # every row has a label

# The label may only be 0 or 1. Anything else would not be a yes/no answer.
assert set(df[TARGET].unique()) == {0, 1}

# --- Summarise each match in one line ----------------------------------------
# groupby("match_id") gathers the rows of each match, then .agg computes:
match_check = df.groupby("match_id").agg(
    rows=(TARGET, "size"),            # how many rows the match has
    positive_labels=(TARGET, "sum"),  # how many of them are labelled 1
    years=("year", "nunique"),        # how many different years appear
)

assert (match_check["rows"] == 2).all()             # exactly two teams per match
assert (match_check["positive_labels"] == 1).all()  # exactly one team supplied the winner
assert (match_check["years"] == 1).all()            # a match belongs to a single tournament

# The same team must not appear twice in the same match.
assert not df.duplicated(subset=["match_id", "Team"]).any()

# --- The two rows must mirror each other -------------------------------------
# If row a says "Russia v Saudi Arabia", row b must say "Saudi Arabia v Russia".
for _, pair in df.groupby("match_id"):
    a, b = pair.iloc[0], pair.iloc[1]
    assert a["Team"] == b["Opponent"]
    assert b["Team"] == a["Opponent"]

print("All 128 matches have two mirrored team rows and exactly one positive label.")

All 128 matches have two mirrored team rows and exactly one positive label.


## 1.3 Missing values and differences between the two tournaments

**What:** count the missing values in every column, then compare how 2018 and 2022 were recorded.

**Why:** the file joins two tournaments that came from different sources. A column that is empty for one whole tournament is useless as a model input, because half the rows would have nothing in it. And if the two years measured a statistic differently, the model could learn "which year is this?" instead of something about football. I need to know both before choosing features.

In [4]:
# --- Missing values per column -----------------------------------------------
# .isna() marks the empty cells, .sum() counts them per column.
missing = df.isna().sum().sort_values(ascending=False)

# Show only the columns that have at least one gap.
display(missing[missing > 0].to_frame("missing_rows"))

# --- Where did each tournament's data come from? -----------------------------
# pd.crosstab counts rows for every (year, value) pair. If a value appears in
# only one year, that column was recorded differently for the two tournaments.
source_columns = ["award_status", "statistics_source", "award_source", "statistics_method"]
for column in source_columns:
    print(f"\n{column}")
    display(pd.crosstab(df["year"], df[column]))

# --- Do the shared statistics look alike in both years? ----------------------
# Mean, smallest and largest value per year for five important stats.
display(
    df.groupby("year")[[
        "Ball Possession %", "Pass Accuracy %", "Passes",
        "Distance Covered (Kms)", "Attempts",
    ]].agg(["mean", "min", "max"]).round(2)
)

,missing_rows
Own goal Time,242
award_winner_player,128
passes_completed,128
on_target_shot_outcomes,128
on_target_fifa_summary,128
incomplete_attempts,128
possession_in_contest_pct,128
fifa_report_url,128
award_winner_player_id,128
1st Goal,73



award_status


award_status,reported,reported_online
year,,
2018,128,0
2022,0,128



statistics_source


statistics_source,FIFA 2018 Statistics.csv,"FIFA post-match report; supplemental match CSV for corners, offsides and fouls"
year,,
2018,128,0
2022,0,128



award_source


award_source,FIFA 2018 Statistics.csv (user supplied),https://www.soccer-blogger.com/2022/12/17/2022-football-world-cup-man-of-the-match-winners-list-of-motm-awards-at-fifa-wc-2022/
year,,
2018,128,0
2022,0,128



statistics_method


statistics_method,FIFA normalized possession excluding in-contest share; computed pass accuracy,Supplied 2018 statistics
year,,
2018,0,128
2022,128,0


Ball Possession %               Pass Accuracy %                Passes  \
                  mean    min    max            mean    min    max    mean   
year                                                                         
2018              50.0  25.00  75.00           82.55  67.00  94.00  462.65   
2022              50.0  15.82  84.18           84.11  65.38  94.53  500.80   

                Distance Covered (Kms)               Attempts          
      min   max                   mean    min    max     mean min max  
year                                                                   
2018  189  1137                 106.66   80.0  148.0    12.59   3  26  
2022  224  1070                 115.25  102.3  148.2    11.21   0  32

### What the audit found

- **The label is balanced, and the sample is 128 matches.** There are 128 ones and 128 zeros because each match gives one of each. The useful count is 128 matches, not 256 rows.
- **Some columns exist only for 2022.** `passes_completed`, `possession_in_contest_pct`, `incomplete_attempts`, `fifa_report_url` and a few others are empty for all 128 rows of 2018. I will not use them.
- **The two years were measured differently.** Every source column has one value for 2018 and a different one for 2022. The 2022 method note says possession was normalised to exclude an "in contest" share and pass accuracy was computed, which is why 2022 possession has decimals (15.82 to 84.18) and 2018 has whole numbers (25 to 75). I keep only statistics that both years record, and I report results for each year separately in Phase 5.
- **Some columns give the answer away.** `award_winner_team` names the team that won the award, so it is the label in another form. `award_winner_player` and `award_winner_player_id` reveal it too. None of them may ever be a model input.

---
# Phase 2: Features and split

## 2.1 Choose the feature columns

**What:** build two lists of input columns ("features"): one with goals and one without.

**Why:**

- **Only shared numeric statistics.** I use the 19 statistics that both tournaments record. I leave out the award columns (they contain the answer), identifiers such as team names and dates (they describe *who* played, not *how*), the source columns and the 2022-only columns.
- **The label is never an input.** `Man of the Match` is the answer key the model learns from during training, and the key I grade with after testing. If it were an input, the model would simply copy it.
- **Goals nearly reveal the answer.** The award usually goes to the side that scored more. A model that sees goals mostly learns "who scored more" and has little reason to look at anything else. So I train every model twice:
  - **with goals:** all 19 statistics. This shows how well the award can be predicted with everything available.
  - **no goals:** drops `Goal Scored`, `Goals in PSO` (shootout goals show who went through) and `Own goals` (part of the scoreline). This is my **main** model. I expect it to score *lower*, because goals make the task easier. I choose it as the main model because it answers the more interesting question: does **how a team played** predict the award?

There is no minutes-played column, so I cannot scale counts to "per 90 minutes". Matches that went to extra time keep their larger raw counts (passes, shots, distance). I keep those matches, because 128 is already a small sample, and I list this as a limitation.

In [5]:
# The 19 numeric statistics that both 2018 and 2022 record.
FEATURES_WITH_GOALS = [
    "Goal Scored", "Ball Possession %", "Attempts", "On-Target",
    "Off-Target", "Blocked", "Corners", "Offsides", "Free Kicks",
    "Saves", "Pass Accuracy %", "Passes", "Distance Covered (Kms)",
    "Fouls Committed", "Yellow Card", "Yellow & Red", "Red",
    "Goals in PSO", "Own goals",
]

# The columns that give away the scoreline, and so nearly give away the answer.
SCORE_LEAK_COLUMNS = ["Goal Scored", "Goals in PSO", "Own goals"]

# The no-goals set is the full set minus the score columns. Building it from
# the first list (instead of typing it again) means the two can never drift apart.
FEATURES_NO_GOALS = [
    column for column in FEATURES_WITH_GOALS if column not in SCORE_LEAK_COLUMNS
]

# A dictionary lets later cells loop over both sets by name.
FEATURE_SETS = {
    "with goals": FEATURES_WITH_GOALS,
    "no goals": FEATURES_NO_GOALS,
}

# --- Safety checks on the lists ----------------------------------------------
for set_name, features in FEATURE_SETS.items():
    assert TARGET not in features               # the label must never be an input
    assert len(features) == len(set(features))  # no column listed twice
    assert set(features).issubset(df.columns)   # every column really exists in the file
    print(f"{set_name}: {len(features)} features")

assert len(FEATURES_WITH_GOALS) == 19 and len(FEATURES_NO_GOALS) == 16
print("Dropped for the no-goals set:", SCORE_LEAK_COLUMNS)

# --- Make sure every feature is a number -------------------------------------
# pd.to_numeric turns text such as "12" into the number 12. errors="coerce"
# turns anything unreadable into NaN (a marked gap) instead of crashing.
# Gaps are filled later, inside each model's pipeline, using training rows only.
df[FEATURES_WITH_GOALS] = df[FEATURES_WITH_GOALS].apply(pd.to_numeric, errors="coerce")

# Count the gaps left after the conversion. All zeros means the data was clean.
display(df[FEATURES_WITH_GOALS].isna().sum().to_frame("missing_after_conversion"))

# No feature may be completely empty, or it would carry no information at all.
assert not df[FEATURES_WITH_GOALS].isna().all().any()

with goals: 19 features
no goals: 16 features
Dropped for the no-goals set: ['Goal Scored', 'Goals in PSO', 'Own goals']


,missing_after_conversion
Goal Scored,0
Ball Possession %,0
Attempts,0
On-Target,0
Off-Target,0
Blocked,0
Corners,0
Offsides,0
Free Kicks,0
Saves,0


## 2.2 Split the matches into training and test sets

**What:** set aside 20% of the **matches** (26 of 128) as a test set, with the same number from each tournament. The other 102 matches are for training.

**Why:**

- **A test set is the only fair exam.** A model can score well on rows it has already seen simply by memorising them. Its score on matches it has never seen is the honest measure.
- **Split by match, not by row.** The two rows of a match mirror each other. If one row went to training and the other to testing, the model would have seen the opponent's side of a test match, which is a way of peeking at the answer. So I pick whole matches, and both rows follow.
- **Stratify by year.** `stratify` keeps the 2018/2022 balance the same on both sides (13 test matches per year). Otherwise the test set could, by chance, be mostly one tournament.
- **A country may appear on both sides.** Brazil can be in a training match and in a different test match. That is fine. It is the *match* that must not be shared.

In [6]:
def split_by_match(frame: pd.DataFrame, test_size: float = 0.20,
                   random_state: int = RANDOM_STATE):
    """Split whole matches (not rows) into a training table and a test table."""
    # One line per match: its id and its tournament year.
    match_meta = frame[["match_id", "year"]].drop_duplicates("match_id")

    # Split the list of match ids, not the rows. stratify keeps the share of
    # 2018 and 2022 matches equal on both sides.
    train_ids, test_ids = train_test_split(
        match_meta["match_id"],
        test_size=test_size,
        random_state=random_state,
        stratify=match_meta["year"],
    )

    # Bring back the team rows. Both rows of a match share its match_id, so
    # they always land on the same side. .copy() gives independent tables, so
    # changing one later can never change the original.
    train = frame[frame["match_id"].isin(train_ids)].copy()
    test = frame[frame["match_id"].isin(test_ids)].copy()
    return train, test


train_df, test_df = split_by_match(df)

# The label columns, held apart from the features.
y_train, y_test = train_df[TARGET], test_df[TARGET]

# --- Check the split did what it promised ------------------------------------
# No match may appear on both sides.
assert set(train_df["match_id"]).isdisjoint(test_df["match_id"])

# 102 training matches and 26 test matches.
assert train_df["match_id"].nunique() == 102 and test_df["match_id"].nunique() == 26

# Two rows per match: 204 and 52 team ROWS (not 204 teams).
assert len(train_df) == 204 and len(test_df) == 52

# Exactly one award side per match on both sides.
assert y_train.sum() == 102 and y_test.sum() == 26

print("Training matches per year:")
print(train_df.groupby("year")["match_id"].nunique())
print("\nTest matches per year:")
print(test_df.groupby("year")["match_id"].nunique())

Training matches per year:
year
2018    51
2022    51
Name: match_id, dtype: int64

Test matches per year:
year
2018    13
2022    13
Name: match_id, dtype: int64


### What the split means for the results

The test set is 26 **matches**, so **one match is worth 1 / 26 ≈ 3.85 percentage points** of accuracy. If two methods differ by one or two matches, that is noise, not a real difference. I keep this in mind when reading every result below.

The 26 test matches are not used again until Phase 5.

---
# Phase 3: Baseline

## 3.1 Score one pick per match

**What:** two helper functions. `match_predictions` takes one score per team row, picks the higher-scoring team in each match and checks the pick against the real label. `match_accuracy` returns the share of matches picked correctly.

**Why:** the question is "which of the two teams?", so there must be exactly **one pick per match**. An ordinary row-by-row accuracy would allow impossible answers, such as predicting `1` for both teams of the same match or `0` for both. Comparing the two scores inside a match forces one winner every time. If the two scores are exactly equal, the alphabetically first team is picked, so the result is the same on every run.

Every method in this notebook (the baseline and all the models) is graded by these two functions, so they are all measured in the same way.

In [7]:
def match_predictions(frame: pd.DataFrame, scores) -> pd.DataFrame:
    """Pick the higher-scoring team in each match and grade the pick.

    Returns one row per match: predicted_team, selected_score, actual_team,
    year and correct (True when the pick matches the real label).
    """
    scores_array = np.asarray(scores, dtype=float)

    # One score per team row is required, or the scores would be matched to
    # the wrong teams.
    if len(scores_array) != len(frame):
        raise ValueError("Need exactly one score per team row.")

    # Work on a small copy so the caller's table is never changed.
    result = frame[["match_id", "Team", "year", TARGET]].copy()
    result["score"] = scores_array

    # Sort inside each match: highest score first, and on an exact tie the
    # alphabetically first team. Keeping the first row of each match is the pick.
    predicted = (
        result.sort_values(["match_id", "score", "Team"], ascending=[True, False, True])
        .drop_duplicates("match_id")[["match_id", "Team", "score"]]
        .rename(columns={"Team": "predicted_team", "score": "selected_score"})
    )

    # The true answer: the row labelled 1 in each match.
    actual = result[result[TARGET] == 1][["match_id", "Team", "year"]].rename(
        columns={"Team": "actual_team"}
    )

    # Put the pick and the truth side by side. validate= raises an error if a
    # match somehow has more than one pick or more than one true answer.
    comparison = predicted.merge(actual, on="match_id", validate="one_to_one")
    comparison["correct"] = comparison["predicted_team"] == comparison["actual_team"]
    return comparison.reset_index(drop=True)


def match_accuracy(frame: pd.DataFrame, scores) -> float:
    """Share of matches where the picked team supplied the Man of the Match."""
    # "correct" is True/False, and the mean of True/False is the share of Trues.
    return float(match_predictions(frame, scores)["correct"].mean())

In [8]:
# Self-check: match-level scoring on tiny hand-made matches.

def tiny_match(match_id, teams, goals, on_target, possession, motm_index, year=2018):
    """Two mirrored team rows for one invented match.

    Each pair such as goals=(1, 0) is (first team, second team).
    motm_index says which of the two teams gets the label 1.
    """
    return [
        {
            "match_id": match_id,
            "year": year,
            "Team": teams[i],
            "Goal Scored": goals[i],
            "On-Target": on_target[i],
            "Ball Possession %": possession[i],
            TARGET: int(i == motm_index),
        }
        for i in (0, 1)
    ]


def tiny_frame(*matches) -> pd.DataFrame:
    """Stack several invented matches into one table."""
    return pd.DataFrame([row for match in matches for row in match])


# Two matches. The award went to A in the first and to D in the second.
check = tiny_frame(
    tiny_match("m1", ("A", "B"), (1, 0), (1, 1), (50, 50), motm_index=0),
    tiny_match("m2", ("C", "D"), (1, 0), (1, 1), (50, 50), motm_index=1),
)
# These scores favour A and C, so the first pick is right and the second is wrong.
graded = match_predictions(check, [0.9, 0.1, 0.8, 0.3])
assert graded["predicted_team"].tolist() == ["A", "C"]
assert graded["correct"].tolist() == [True, False]
assert match_accuracy(check, [0.9, 0.1, 0.8, 0.3]) == 0.5

# An exact tie goes to the alphabetically first team (Angola before Zambia).
tie = tiny_frame(tiny_match("m1", ("Zambia", "Angola"), (0, 0), (1, 1), (50, 50), 1))
assert match_predictions(tie, [0.5, 0.5])["predicted_team"].tolist() == ["Angola"]

# The wrong number of scores must raise an error instead of guessing.
try:
    match_predictions(tie, [0.5])
    raise AssertionError("Expected a ValueError for a missing score.")
except ValueError:
    pass

print("Self-check passed: match-level scoring.")

Self-check passed: match-level scoring.


## 3.2 The baseline rule

**What:** a hand-written rule that picks a team **without any training**:

1. the team with **more goals**;
2. if goals are level, the team with **more shots on target**;
3. if those are level too, the team with **more possession**;
4. if everything is level, a **coin flip with a fixed seed**, so the rule always gives an answer and gives the same one on every run.

**Why:** a model's accuracy means nothing on its own. Is 70% good? That depends on what a simple rule can already do. The baseline is that yardstick: it is not a training stage, and it learns nothing. A model earns its place only if it beats the rule. (Picking at random would get about 50%.)

The team the rule picks is its *predicted Man of the Match side*, which is not always the match winner. The tie-break chain belongs to the baseline only. The models in Phase 4 learn for themselves how much each statistic matters.

In [9]:
# The baseline compares these columns in this order. The first one that
# differs between the two teams decides the pick.
BASELINE_ORDER = ["Goal Scored", "On-Target", "Ball Possession %"]


def baseline_scores(frame: pd.DataFrame, seed: int = RANDOM_STATE) -> np.ndarray:
    """Hand-written rule: returns 1 for the team it picks in each match, 0 for the other."""
    # One random number per row: the coin flip of last resort. A generator
    # built from a fixed seed produces the same numbers on every run.
    coin = np.random.default_rng(seed).random(len(frame))

    ranked = frame[["match_id", *BASELINE_ORDER]].copy()
    ranked["_coin"] = coin
    ranked["_row"] = np.arange(len(frame))  # remember where each row sits in the table

    # Sorting by several columns compares them one after another, the way a
    # dictionary sorts words letter by letter: goals first, then shots on
    # target only if goals are level, then possession, then the coin.
    # After the sort, the LAST row of each match is the team that ranks highest.
    ranked = ranked.sort_values(["match_id", *BASELINE_ORDER, "_coin"])
    picked_rows = ranked.groupby("match_id")["_row"].last().to_numpy()

    # Score 1 for the picked row and 0 for the other, in the original row order.
    scores = np.zeros(len(frame))
    scores[picked_rows] = 1.0
    return scores


def drawn_match_ids(frame: pd.DataFrame) -> list:
    """The ids of matches where both teams scored the same number of goals."""
    # If the smallest and the largest goal count in a match are equal, it was level.
    goals = frame.groupby("match_id")["Goal Scored"].agg(["min", "max"])
    return goals.index[goals["min"] == goals["max"]].tolist()

In [10]:
# Self-check: the baseline's tie-break chain, one step at a time.

def baseline_picks(data: pd.DataFrame, seed: int = RANDOM_STATE) -> list:
    """The team the baseline picks in each match, in match order."""
    picks = match_predictions(data, baseline_scores(data, seed))
    return picks.sort_values("match_id")["predicted_team"].tolist()


# Step 1: more goals wins, even though A had more shots and more possession.
assert baseline_picks(tiny_frame(tiny_match("m1", ("A", "B"), (0, 2), (9, 1), (70, 30), 1))) == ["B"]

# Step 2: goals level (1-1), so more shots on target decides.
assert baseline_picks(tiny_frame(tiny_match("m1", ("A", "B"), (1, 1), (2, 6), (70, 30), 1))) == ["B"]

# Step 3: goals and shots level, so more possession decides.
assert baseline_picks(tiny_frame(tiny_match("m1", ("A", "B"), (1, 1), (4, 4), (45, 55), 1))) == ["B"]

# Step 4: twenty matches that are level on everything. The coin must pick
# exactly one team per match, give the same picks on a rerun, and not always
# land on the same side.
all_level = tiny_frame(*[
    tiny_match(f"m{i}", (f"A{i}", f"B{i}"), (0, 0), (3, 3), (50, 50), 0)
    for i in range(20)
])
coin_scores = baseline_scores(all_level, seed=7)
assert coin_scores.reshape(-1, 2).sum(axis=1).tolist() == [1.0] * 20
assert baseline_picks(all_level, seed=7) == baseline_picks(all_level, seed=7)
assert len({team[0] for team in baseline_picks(all_level, seed=7)}) == 2

# Shuffling the rows must not change which teams the rule picks.
two_matches = tiny_frame(
    tiny_match("m1", ("A", "B"), (1, 1), (2, 6), (70, 30), 1),
    tiny_match("m2", ("C", "D"), (3, 0), (5, 5), (40, 60), 0),
)
shuffled = two_matches.iloc[[3, 0, 2, 1]].reset_index(drop=True)
assert sorted(baseline_picks(two_matches)) == sorted(baseline_picks(shuffled)) == ["B", "C"]

# drawn_match_ids finds the 1-1 match and ignores the 3-0 match.
assert drawn_match_ids(two_matches) == ["m1"]

print("Self-check passed: baseline rule.")

Self-check passed: baseline rule.


## 3.3 The baseline on the training matches

**What:** run the rule on the 102 training matches, and look separately at the drawn ones.

**Why:** this gives a first idea of how strong the rule is without touching the test set. Drawn matches matter most: goals cannot decide them, so the rule has to fall back on its weaker tie-breaks. They are where a model has the best chance of adding something.

In [11]:
# Grade the rule on every training match.
baseline_train = match_predictions(train_df, baseline_scores(train_df))

# The training matches that ended level, and the rule's results on just those.
train_draws = drawn_match_ids(train_df)
on_draws = baseline_train[baseline_train["match_id"].isin(train_draws)]

print(f"Baseline on all training matches: {baseline_train['correct'].mean():.3f}")
print(f"Baseline on drawn training matches: {on_draws['correct'].sum()}/{len(on_draws)}")
print("Random-choice expectation: 0.500")

Baseline on all training matches: 0.902
Baseline on drawn training matches: 12/20
Random-choice expectation: 0.500


The rule is right on about 90% of training matches, so goals alone carry most of the answer. On the 20 drawn matches it gets 12, which is close to a coin flip.

The baseline's **test** score is computed in Phase 5, on the same 26 matches as the models and with the draws included. Leaving the draws out would let the rule skip its hardest cases and look better than it is.

---
# Phase 4: Training and tuning

## 4.1 Three models, from simple to more flexible

**What:** define three models. Each one is wrapped in a `Pipeline`, which runs the preparation steps and then the model as a single object.

**Why these three:** they form a progression from simple to more flexible. That lets me test whether extra flexibility actually helps on a dataset this small.

| Model | How it decides | What it is good for |
|---|---|---|
| **Logistic regression** | Multiplies each statistic by a learned weight, adds them up and squashes the total into a probability between 0 and 1 | The simplest of the three, least likely to memorise, and its weights can be read |
| **Decision tree** | Asks a chain of yes/no questions, such as "more than 5 shots on target?" | Easy to draw as a flowchart, but a deep tree memorises small data |
| **KNN** (k-nearest neighbours) | Finds the `k` most similar training rows and lets them vote | No formula at all: it only assumes that similar matches end alike |

**Why not linear regression?** It predicts any number, including values below 0 or above 1. Logistic regression squashes its output into the 0 to 1 range, which is what "how likely is this team?" needs.

**Why a pipeline, and why scaling:**

- **Imputer.** Fills any missing value with the column's median. The data has no gaps today, but the step keeps the models from crashing if a gap ever appears.
- **Scaler.** For each column, it subtracts the column's mean and divides by its spread, so every statistic ends up on a similar scale. It works down each *column*, one statistic at a time.
  - KNN **needs** it. KNN measures distance between rows. Passes are in the hundreds while cards are 0 to 3, so without scaling the passes would drown out everything else.
  - Logistic regression **wants** it, so that its penalty treats every statistic fairly and its weights can be compared with each other.
  - The tree **does not need** it. A tree only asks "is this statistic above a threshold?", and rescaling a column moves the threshold without changing the split.
- **Inside the pipeline.** Because the imputer and scaler are pipeline steps, they are fitted only on the rows the model is training on. Fitting them on all the data would leak a little information from the rows being scored.

In [12]:
def build_models() -> dict:
    """Return the three untrained models, each inside its own Pipeline."""
    return {
        # Model 1: logistic regression. max_iter=2000 gives the solver enough
        # steps to finish; random_state keeps the result reproducible.
        "Logistic regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),  # fill gaps with the median
            ("scaler", StandardScaler()),                   # same scale for every stat
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        # Model 2: decision tree. No scaler, because a tree only compares a
        # statistic with a threshold, so scaling would not change its splits.
        "Decision tree": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]),
        # Model 3: KNN. The scaler is required here, because KNN compares
        # distances and unscaled passes (hundreds) would drown out cards (0 to 3).
        "KNN": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier()),
        ]),
    }


def positive_scores(fitted: Pipeline, features: pd.DataFrame) -> np.ndarray:
    """The model's score for label 1 ("the award came from this team") on each row."""
    # predict_proba returns one column per label. classes_ lists the labels in
    # column order, so this finds the column that belongs to label 1.
    positive_column = list(fitted.classes_).index(1)
    return fitted.predict_proba(features)[:, positive_column]

## 4.2 The settings to try

**What:** for each model, a short list of values for its main setting (its "hyperparameter"), plus a rule that says which of two settings is the simpler one.

**Why:** a setting controls how flexible a model is. If it is too rigid, the model misses real patterns. If it is too flexible, it **memorises** the training matches ("overfits") and fails on new ones. I do not know the right value in advance, so I give a range and let cross-validation choose.

| Model | Setting | Values | Meaning |
|---|---|---|---|
| Logistic regression | `C` | 0.01, 0.1, 1, 10, 100 | Small `C` keeps the weights small (simpler). Large `C` lets the model fit the training data more closely. `C` can be any positive number; it is the model's *output* that lies between 0 and 1. |
| Decision tree | `max_depth` | 2, 3, 4, 5, 6 | How many questions the tree may ask in a row. |
| Decision tree | `min_samples_leaf` | 2, 5, 10 | The fewest rows allowed in a final group. A larger value stops the tree from writing a rule for one or two odd matches. |
| KNN | `n_neighbors` (k) | 3, 5, 7, 9, 11, 13, 15 | How many neighbours vote. Odd numbers avoid a tied vote. A larger k gives a smoother, simpler decision. |

**Tie rule.** If two settings get exactly the same cross-validation score, I keep the **simpler** one: smaller `C`, shallower tree with larger leaves, larger k. A simpler model is easier to explain and less likely to have memorised the data.

In [13]:
# The values to try for each model. The "model__" prefix tells the Pipeline
# that the setting belongs to its step named "model".
GRIDS = {
    "Logistic regression": {"model__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
    "Decision tree": {
        "model__max_depth": [2, 3, 4, 5, 6],
        "model__min_samples_leaf": [2, 5, 10],
    },
    "KNN": {"model__n_neighbors": [3, 5, 7, 9, 11, 13, 15]},
}


def simplicity_key(model_name: str, params: dict) -> tuple:
    """A sort key where a lower value means a simpler model.

    Used only to break an exact tie in cross-validation scores.
    """
    if model_name == "Logistic regression":
        # A smaller C means stronger regularisation, which is simpler.
        return (params["model__C"],)
    if model_name == "Decision tree":
        # A shallower tree is simpler. At equal depth, bigger leaves are
        # simpler, hence the minus sign (a bigger leaf gives a lower key).
        return (params["model__max_depth"], -params["model__min_samples_leaf"])
    if model_name == "KNN":
        # More neighbours give a smoother, simpler decision, hence the minus sign.
        return (-params["model__n_neighbors"],)
    raise ValueError(f"Unknown model: {model_name}")

In [14]:
# Self-check: the grids and the pipelines are set up as described above.

# Every model has a grid, and every grid belongs to a model.
assert set(GRIDS) == set(build_models())

# The scaler sits inside the two models that need it, and not in the tree.
models_check = build_models()
assert "scaler" in models_check["Logistic regression"].named_steps
assert "scaler" in models_check["KNN"].named_steps
assert "scaler" not in models_check["Decision tree"].named_steps

# "Simpler" means: smaller C, a shallower tree, a larger k.
assert simplicity_key("Logistic regression", {"model__C": 0.1}) < simplicity_key(
    "Logistic regression", {"model__C": 10.0})
assert simplicity_key("Decision tree", {"model__max_depth": 2, "model__min_samples_leaf": 5}) < simplicity_key(
    "Decision tree", {"model__max_depth": 4, "model__min_samples_leaf": 5})
assert simplicity_key("KNN", {"model__n_neighbors": 15}) < simplicity_key(
    "KNN", {"model__n_neighbors": 3})

print("Self-check passed: grids and pipelines.")

Self-check passed: grids and pipelines.


## 4.3 Grouped 5-fold cross-validation

**What:** a way to score every setting using the training matches only.

1. Divide the 102 training matches into **5 groups** ("folds") of about 20 matches. Both rows of a match stay in the same fold.
2. Hold out **one** fold for checking and train on the other **four**.
3. Rotate until every fold has been the held-out one exactly once. That is 5 rounds.
4. **Average** the 5 scores. That average is the setting's cross-validation (CV) score.
5. Repeat for every setting of every model.

**Why:**

- **The test set must stay unseen.** If I chose settings by looking at test scores, I would be tuning to the test matches, and they would stop being a fair exam. Cross-validation creates practice exams out of the training data instead.
- **Why 5 rounds and not 1.** A single held-out fold of 20 matches is very noisy. Averaging 5 rounds uses every training match for checking once, which gives a steadier number.
- **Why grouped.** It is the same reason as in the split. If a match's two rows were in different folds, the model would see the opponent's side of a match it is being checked on.
- **Why match accuracy.** Each fold is scored with `match_accuracy`, the same measure used for the final result, so the settings are chosen for the thing I actually care about.

In [15]:
def grouped_cv_search(train: pd.DataFrame, features: list, n_splits: int = 5) -> pd.DataFrame:
    """Score every setting of every model with grouped cross-validation.

    Returns one row per (model, setting) with the mean and spread of its fold scores.
    Only training rows are used. The test set is never touched here.
    """
    X = train[features]  # the inputs
    y = train[TARGET]    # the answers

    # Build the 5 folds once, so that every model and every setting is checked
    # on exactly the same folds (a fair comparison). groups=match_id is what
    # keeps both rows of a match in the same fold.
    folds = list(GroupKFold(n_splits=n_splits).split(X, y, groups=train["match_id"]))

    rows = []
    for model_name, base_model in build_models().items():
        # ParameterGrid lists every combination of the values in the grid.
        for params in ParameterGrid(GRIDS[model_name]):
            fold_scores = []
            for fit_idx, valid_idx in folds:
                # clone gives a fresh, untrained copy, so nothing learned in
                # one fold can carry over into the next.
                candidate = clone(base_model).set_params(**params)

                # Train on four folds...
                candidate.fit(X.iloc[fit_idx], y.iloc[fit_idx])

                # ...then score the held-out fold, one pick per match.
                scores = positive_scores(candidate, X.iloc[valid_idx])
                fold_scores.append(match_accuracy(train.iloc[valid_idx], scores))

            rows.append({
                "model": model_name,
                "params": params,
                "cv_mean": float(np.mean(fold_scores)),  # the setting's CV score
                "cv_std": float(np.std(fold_scores)),    # how much the 5 folds disagree
                "fold_scores": [round(score, 3) for score in fold_scores],
            })
    return pd.DataFrame(rows)

## 4.4 Pick the best setting and refit

**What:** `best_settings` keeps the top-scoring setting for each model, using the tie rule from 4.2. `fit_best` then trains each model again with its winning setting on **all 102** training matches.

**Why refit:** during cross-validation each model only ever trained on four fifths of the training data. Once the setting is chosen there is no reason to waste the remaining fifth. The final model should learn from every training match.

In [16]:
def best_settings(search_results: pd.DataFrame) -> pd.DataFrame:
    """For each model, the setting with the highest CV mean (ties go to the simpler one)."""
    winners = []
    for model_name, rows in search_results.groupby("model"):
        # Every setting that reaches the top score. np.isclose compares
        # decimals safely, because 0.7 and 0.7000000001 should count as equal.
        top = rows[np.isclose(rows["cv_mean"], rows["cv_mean"].max())]

        # Among the tied settings, keep the simplest one.
        simplest = min(
            top.to_dict("records"),
            key=lambda row: simplicity_key(model_name, row["params"]),
        )
        winners.append(simplest)

    # Best model first.
    return (
        pd.DataFrame(winners)
        .sort_values("cv_mean", ascending=False, kind="stable")
        .reset_index(drop=True)
    )


def fit_best(train: pd.DataFrame, features: list, best: pd.DataFrame) -> dict:
    """Train each model with its winning setting on all the training matches."""
    models = build_models()
    fitted = {}
    for row in best.itertuples():
        # A fresh copy of the model with the winning setting switched on.
        pipeline = clone(models[row.model]).set_params(**row.params)
        fitted[row.model] = pipeline.fit(train[features], train[TARGET])
    return fitted

In [17]:
# Self-check: an exact tie in CV score goes to the simpler setting.
tied_results = pd.DataFrame([
    {"model": "KNN", "params": {"model__n_neighbors": 3}, "cv_mean": 0.7},
    {"model": "KNN", "params": {"model__n_neighbors": 11}, "cv_mean": 0.7},
    {"model": "KNN", "params": {"model__n_neighbors": 5}, "cv_mean": 0.6},
])
# k=3 and k=11 tie on 0.7. The larger k is simpler, so k=11 must win.
assert best_settings(tied_results).loc[0, "params"] == {"model__n_neighbors": 11}

print("Self-check passed: tie-breaking picks the simpler setting.")

Self-check passed: tie-breaking picks the simpler setting.


## 4.5 Run the tuning for both feature sets

**What:** run the search, pick the winners and refit, once for the "with goals" features and once for the "no goals" features. Then note which model has the best CV score in each set.

**Why:** the model for each feature set is chosen here, from cross-validation, **before** the test matches are looked at. Choosing it after seeing the test scores would be picking the winner with the answers in hand.

In [18]:
searches = {}       # every setting's CV score, per feature set
best_by_set = {}    # the winning setting of each model, per feature set
fitted_by_set = {}  # the final trained models, per feature set

for set_name, features in FEATURE_SETS.items():
    searches[set_name] = grouped_cv_search(train_df, features)
    best_by_set[set_name] = best_settings(searches[set_name])
    fitted_by_set[set_name] = fit_best(train_df, features, best_by_set[set_name])

    print(f"\nBest settings ({set_name}):")
    display(best_by_set[set_name][["model", "params", "cv_mean", "cv_std", "fold_scores"]])

# The tables are sorted best first, so row 0 is the model CV prefers.
selected = {set_name: best.iloc[0]["model"] for set_name, best in best_by_set.items()}
print("Selected by grouped CV:", selected)


Best settings (with goals):


,model,params,cv_mean,cv_std,fold_scores
0,Logistic regression,{'model__C': 0.01},0.873810,0.115116,"[0.667, 0.952, 0.85, 0.9, 1.0]"
1,Decision tree,"{'model__max_depth': 2, 'model__min_samples_leaf': 10}",0.832857,0.065990,"[0.762, 0.952, 0.8, 0.8, 0.85]"
2,KNN,{'model__n_neighbors': 11},0.774286,0.049611,"[0.714, 0.857, 0.75, 0.8, 0.75]"



Best settings (no goals):


,model,params,cv_mean,cv_std,fold_scores
0,Logistic regression,{'model__C': 10.0},0.688095,0.111433,"[0.476, 0.714, 0.75, 0.7, 0.8]"
1,KNN,{'model__n_neighbors': 3},0.674762,0.146830,"[0.619, 0.905, 0.7, 0.7, 0.45]"
2,Decision tree,"{'model__max_depth': 3, 'model__min_samples_leaf': 10}",0.598095,0.159288,"[0.381, 0.81, 0.7, 0.45, 0.65]"


Selected by grouped CV: {'with goals': 'Logistic regression', 'no goals': 'Logistic regression'}


## 4.6 How the CV score changes across settings

**What:** for the no-goals set (the main one), list each model's five best settings.

**Why:** if the scores barely move from one setting to the next, the exact setting does not matter much and the model is not sensitive to it. Large jumps between neighbouring settings would be a warning that the choice is partly luck.

In [19]:
# Turn the settings into text so they print on one line.
grid_view = searches["no goals"].assign(params=lambda table: table["params"].astype(str))

for model_name, rows in grid_view.groupby("model"):
    print(model_name)
    display(
        rows[["params", "cv_mean", "cv_std"]]
        .sort_values("cv_mean", ascending=False)
        .head(5)
    )

Decision tree


,params,cv_mean,cv_std
10,"{'model__max_depth': 3, 'model__min_samples_leaf': 10}",0.598095,0.159288
15,"{'model__max_depth': 5, 'model__min_samples_leaf': 5}",0.596667,0.144150
13,"{'model__max_depth': 4, 'model__min_samples_leaf': 10}",0.588095,0.183861
5,"{'model__max_depth': 2, 'model__min_samples_leaf': 2}",0.586190,0.140338
6,"{'model__max_depth': 2, 'model__min_samples_leaf': 5}",0.586190,0.140338


KNN


,params,cv_mean,cv_std
20,{'model__n_neighbors': 3},0.674762,0.146830
21,{'model__n_neighbors': 5},0.619048,0.159385
22,{'model__n_neighbors': 7},0.608095,0.107165
23,{'model__n_neighbors': 9},0.558571,0.095447
24,{'model__n_neighbors': 11},0.558571,0.070832


Logistic regression


,params,cv_mean,cv_std
3,{'model__C': 10.0},0.688095,0.111433
4,{'model__C': 100.0},0.688095,0.111433
2,{'model__C': 1.0},0.678571,0.129678
1,{'model__C': 0.1},0.658571,0.116444
0,{'model__C': 0.01},0.628571,0.090213


### Reading the tuning results

- **Logistic regression has the best CV score in both feature sets**, so it is the selected model for both. That makes *logistic regression (no goals)* the main model.
- **Removing goals costs a lot.** The best CV score falls from about 0.87 with goals to about 0.69 without. As expected, goals make the task easier.
- **Even with goals, no model's CV score beats the baseline's 0.90** on the training matches.
- **`cv_std` is the spread between the 5 folds.** It is not a confidence interval. Each fold has only about 20 matches, so one match moves a fold's score by about 5 points. The no-goals models show spreads of 0.11 to 0.16, which is a lot of disagreement between folds.

---
# Phase 5: Evaluation

## 5.1 Final scores on the 26 test matches

**What:** grade the baseline and all six tuned models (three models × two feature sets) on the 26 test matches.

**Why:** this is the first and only time the test matches are used. Every method is graded on the same 26 matches by the same function, so the comparison is fair. Whatever the numbers turn out to be, I do not go back and adjust anything. Tuning after seeing a test score would turn the test set into more training data.

In [20]:
def summarise(name: str, comparison: pd.DataFrame) -> dict:
    """One line of the results table: correct matches out of the total."""
    return {
        "method": name,
        "correct_matches": int(comparison["correct"].sum()),
        "total_matches": len(comparison),
        "match_accuracy": round(float(comparison["correct"].mean()), 3),
    }


# Start with the baseline rule, graded on the test matches (draws included).
predictions = {"Baseline rule": match_predictions(test_df, baseline_scores(test_df))}

# Then every tuned model, for both feature sets.
for set_name, fitted_models in fitted_by_set.items():
    # A model must be given exactly the columns it was trained on.
    X_test = test_df[FEATURE_SETS[set_name]]
    for model_name, fitted in fitted_models.items():
        scores = positive_scores(fitted, X_test)
        predictions[f"{model_name} ({set_name})"] = match_predictions(test_df, scores)

results_table = pd.DataFrame(
    [summarise(name, comparison) for name, comparison in predictions.items()]
)
display(results_table)

# A reminder of how coarse the scale is with only 26 matches.
print("One test match =", round(100 / 26, 2), "percentage points")

,method,correct_matches,total_matches,match_accuracy
0,Baseline rule,21,26,0.808
1,Logistic regression (with goals),19,26,0.731
2,Decision tree (with goals),18,26,0.692
3,KNN (with goals),17,26,0.654
4,Logistic regression (no goals),18,26,0.692
5,KNN (no goals),11,26,0.423
6,Decision tree (no goals),13,26,0.500


One test match = 3.85 percentage points


## 5.2 Results by tournament and on drawn matches

**What:** split each method's correct picks into 2018 and 2022 (13 test matches each), and count its correct picks on the drawn test matches.

**Why:**

- **By year.** Phase 1 showed the two tournaments were measured differently. If a method does much better in one year, that is worth knowing. With 13 matches per year it is a hint only.
- **On draws.** When the goals are level, the scoreline cannot decide. This is where a model that looks at how the teams played has its best chance to beat the rule.

In [21]:
# The test matches where both teams scored the same number of goals.
test_draws = drawn_match_ids(test_df)


def breakdown(comparison: pd.DataFrame) -> dict:
    """Correct picks per tournament year and on drawn matches."""
    by_year = comparison.groupby("year")["correct"].sum()
    drawn = comparison[comparison["match_id"].isin(test_draws)]["correct"]
    return {
        "2018 (/13)": int(by_year.get(2018, 0)),
        "2022 (/13)": int(by_year.get(2022, 0)),
        f"draws (/{len(test_draws)})": int(drawn.sum()),
    }


# .T flips the table so that each method is a row.
display(pd.DataFrame({name: breakdown(c) for name, c in predictions.items()}).T)

,2018 (/13),2022 (/13),draws (/8)
Baseline rule,11,10,4
Logistic regression (with goals),11,8,3
Decision tree (with goals),10,8,2
KNN (with goals),8,9,3
Logistic regression (no goals),11,7,3
KNN (no goals),6,5,2
Decision tree (no goals),6,7,3


## 5.3 Where the main model went wrong

**What:** list the test matches that the main model (the no-goals model chosen by CV) picked wrongly.

**Why:** a single accuracy number hides *which* matches were missed. Looking at them shows whether the mistakes have something in common.

`selected_score` is the model's score for the team it picked. It is used to **rank** the two teams within a match. It is not a calibrated probability that the team wins the award: each row is scored separately, so the two scores in a match do not have to add up to 1.

In [22]:
# The main model: the one cross-validation selected for the no-goals features.
main_name = f"{selected['no goals']} (no goals)"
main_comparison = predictions[main_name]
print("Main model:", main_name)

# ~ means "not", so this keeps only the matches the model got wrong.
display(main_comparison.loc[~main_comparison["correct"]].sort_values("match_id"))

Main model: Logistic regression (no goals)


,match_id,predicted_team,selected_score,actual_team,year,correct
4,M-2018-21,Australia,0.614219,Denmark,2018,False
7,M-2018-29,Sweden,0.863793,Germany,2018,False
15,M-2022-06,Denmark,0.643910,Tunisia,2022,False
16,M-2022-14,South Korea,0.257570,Uruguay,2022,False
21,M-2022-42,Belgium,0.556190,Croatia,2022,False
22,M-2022-47,Brazil,0.800715,Cameroon,2022,False
23,M-2022-50,Australia,0.556769,Argentina,2022,False
25,M-2022-57,Brazil,0.950444,Croatia,2022,False


## 5.4 Which statistics the main model leaned on

**What:** two views of what the main model learned.

- **Permutation importance.** Shuffle one statistic's column so that it no longer lines up with the right rows, then see how much the model's score drops. A big drop means the model relied on that statistic.
- **Logistic regression weights.** The number each (scaled) statistic is multiplied by. A positive weight pushes towards "the award came from this team", and a negative weight pushes away from it.

**Why:** a prediction is more useful if I can say what it is based on. Both views are measured on **training data only**, so the test matches are not used to learn anything.

**How far to trust it:** this part is exploratory. Statistics that move together (attempts, shots on target, corners) share importance between them, so no single number is exact. A weight shows an association once the other statistics are held fixed. It does not show cause.

In [23]:
main_features = FEATURES_NO_GOALS
main_model = fitted_by_set["no goals"][selected["no goals"]]

# Shuffle each statistic 20 times and average the drop in score, so that one
# lucky or unlucky shuffle does not decide the ranking. balanced_accuracy is
# a row-level score, which is enough for ranking the statistics.
importance = permutation_importance(
    main_model, train_df[main_features], y_train,
    scoring="balanced_accuracy", n_repeats=20, random_state=RANDOM_STATE,
)
display(
    pd.DataFrame({
        "feature": main_features,
        "mean_importance": importance.importances_mean.round(3),  # average drop in score
        "std_importance": importance.importances_std.round(3),    # spread over the 20 shuffles
    })
    .sort_values("mean_importance", ascending=False)
    .head(10)
)

# The logistic regression weights for the no-goals set. The model was trained
# on scaled statistics, so the weights can be compared with each other.
lr = fitted_by_set["no goals"]["Logistic regression"]
display(
    pd.Series(lr.named_steps["model"].coef_[0], index=main_features)
    .sort_values()
    .round(3)
    .to_frame("standardized_coefficient")
)

,feature,mean_importance,std_importance
2,On-Target,0.126,0.029
0,Ball Possession %,0.054,0.019
4,Blocked,0.022,0.023
6,Offsides,0.021,0.014
1,Attempts,0.020,0.018
7,Free Kicks,0.015,0.018
5,Corners,0.013,0.019
13,Yellow Card,0.013,0.014
10,Passes,0.002,0.013
15,Red,0.002,0.003


,standardized_coefficient
Ball Possession %,-0.797
Blocked,-0.447
Red,-0.362
Offsides,-0.346
Yellow Card,-0.322
Attempts,-0.322
Distance Covered (Kms),-0.116
Yellow & Red,-0.035
Fouls Committed,0.042
Off-Target,0.101


---
# Results and conclusions

## Method in brief

I used the statistics shared by the 2018 and 2022 World Cups to predict which of the two teams in a finished match supplied the Man of the Match. I split by match into 102 training and 26 test matches, with 13 test matches from each tournament. A hand-written rule (goals, then shots on target, then possession, then a seeded coin flip) served as the baseline. Three models (logistic regression, a decision tree and KNN) were tuned with grouped 5-fold cross-validation on the training matches, once with goals and once without. Everything was scored once on the test matches.

## Results on the 26 test matches

| Method | With goals | No goals |
|---|---|---|
| Baseline rule | 21/26 (80.8%) | n/a |
| Logistic regression | 19/26 (73.1%) | **18/26 (69.2%)** (main model) |
| Decision tree | 18/26 (69.2%) | 13/26 (50.0%) |
| KNN | 17/26 (65.4%) | 11/26 (42.3%) |

## What the results say

1. **No model beat the baseline.** The simple rule got 21 of 26. The best model, logistic regression with goals, got 19. Two matches is within the noise of a 26-match test (one match is 3.85 points), so I cannot say the rule is truly better. I can say that the models added nothing measurable on top of it.
2. **Goals carry most of the answer.** With goals, every model reached 17 to 19 of 26. The extra statistics did not improve on "pick the team that scored more".
3. **How a team played still says something.** The main model never saw a goal and still picked 18 of 26, where a coin flip would expect 13. Its strongest signal was **shots on target** (the highest permutation importance and the largest positive weight). With only 26 matches this is an encouraging hint, not proof.
4. **More flexibility did not help.** Without goals, the decision tree (13/26) and KNN (11/26) did no better than a coin flip. KNN looked good in cross-validation (0.67) and then scored 0.42 on the test matches, which shows how unstable results are on data this small. The simplest model was the most reliable one.
5. **Drawn matches stayed hard for everyone.** On the 8 drawn test matches the baseline got 4 and the main model got 3. The no-goals model did not beat the rule where I thought it had the best chance.
6. **2018 went better than 2022 for the main model** (11/13 against 7/13). The two tournaments were measured differently, and 13 matches per year is very few, so I do not read a cause into this gap.

One weight needs care: possession has a clearly **negative** weight in the main model. That does not mean having the ball hurts a team. It means that, among teams with the same shots, passes and corners, the one with more possession was less often the award side. It is an association inside a small, correlated dataset.

## Limitations

- **Small test set.** 26 matches, so one match moves accuracy by 3.85 points. Gaps of one or two matches are noise.
- **Two different sources.** Shared column names do not guarantee that 2018 and 2022 measured the same thing in the same way (possession is the clear example).
- **Label sources.** The 2018 award labels come from a supplied statistics file and the 2022 labels from an online list. They should be checked against official records before being used for anything beyond coursework.
- **No per-90 scaling.** Matches with extra time have larger raw counts, and there is no minutes column to correct for it.
- **After the match, team only.** The model needs full-time statistics, so it cannot forecast before kick-off, and it picks the team, not the player.

## What I would do next

- Add more tournaments recorded in a consistent way. More matches is the only real cure for the noise.
- Check how each statistic was defined in each year before trusting year-by-year comparisons.
- As a separate diagnostic, train on 2018 and test on 2022 to see how well the model transfers between tournaments.